<a href="https://colab.research.google.com/github/s-chudmunge/headlinegpt/blob/main/HeadlineGPT_Training_and_Export.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

### Environment Check
We begin by checking the GPU status using `nvidia-smi`.

In [ ]:
!nvidia-smi

Sun Aug  9 03:58:51 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   40C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

### Dependencies
We install the necessary libraries for fine-tuning, including `transformers`, `peft`, and `trl`.

In [ ]:
!pip -q install -U transformers datasets peft trl bitsandbytes accelerate sentencepiece

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 11.6/11.6 MB 116.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 559.1/559.1 kB 48.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 775.8/775.8 kB 61.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 889.0/889.0 kB 67.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 40.9/40.9 MB 23.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 50.1/50.1 MB 15.3 MB/s eta 0:00:00


### Library Imports
Importing core libraries and checking versions to ensure environment stability.

In [ ]:
import torch
import transformers
import datasets
import peft
import trl

print("Torch:", torch.__version__)
print("Transformers:", transformers.__version__)
print("Datasets:", datasets.__version__)
print("PEFT:", peft.__version__)
print("TRL:", trl.__version__)
print("CUDA:", torch.cuda.is_available())

### Model Initialization
Loading the base Qwen2.5 model with 4-bit quantization config to save memory.

In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM, BitsAndBytesConfig
import torch

MODEL_NAME = "Qwen/Qwen2.5-1.5B-Instruct"

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

model = AutoModelForCausalLM.from_pretrained(
    MODEL_NAME,
    quantization_config=bnb_config,
    device_map="auto",
)

print("Loaded!")

### Storage Setup
Mounting Google Drive to access our training dataset.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

### Data Discovery
Locating the JSONL dataset file within the mounted drive.

In [ ]:
!find /content/drive -name "master_training_dataset.jsonl"

### Schema Analysis
Scanning the dataset to identify all available fields and data types.

In [ ]:
import json

fields = {}

with open("/content/drive/MyDrive/colab_data/master_training_dataset.jsonl") as f:
    for i, line in enumerate(f):
        obj = json.loads(line)

        def scan(d, prefix=""):
            for k, v in d.items():
                name = f"{prefix}.{k}" if prefix else k
                if isinstance(v, dict):
                    scan(v, name)
                else:
                    t = type(v).__name__
                    fields.setdefault(name, set()).add(t)

        scan(obj)

print(fields)

### Dataset Loading
Loading the data into the Hugging Face `datasets` format with a defined schema.

In [ ]:
from datasets import load_dataset, Features, Value

features = Features({
    "content": Value("string"),
    "title": Value("string"),
    "metrics": {
        "impressions": Value("int64"),
        "clicks": Value("int64"),
        "ctr": Value("float64"),
        "claps": Value("int64"),
        "responses": Value("int64"),
        "score": Value("int64"),
        "comments": Value("int64"),
        "views": Value("int64"),
        "likes": Value("int64"),
        "shares": Value("int64"),
        "facebook": Value("int64"),
        "linkedin": Value("int64"),
    },
    "metadata": {
        "platform": Value("string"),
        "test_id": Value("string"),
        "is_winner": Value("bool"),
        "video_id": Value("string"),
        "channel": Value("string"),
        "topic": Value("string"),
        "source": Value("string"),
    },
    "reward_score": Value("float64"),
})

dataset = load_dataset(
    "json",
    data_files="/content/drive/MyDrive/colab_data/master_training_dataset.jsonl",
    features=features,
    split="train",
)

print(dataset)
print(dataset[0])

### Data Splitting
Dividing the dataset into training and validation sets (90/10 split).

In [ ]:
from datasets import DatasetDict

# Train / Validation split
splits = dataset.train_test_split(
    test_size=0.1,
    seed=42
)

train_ds = splits["train"]
val_ds = splits["test"]

print(train_ds)
print(val_ds)

### Prompt Formatting
Applying a chat template to convert raw content and titles into instructions for the LLM.

In [ ]:
def format_example(example):
    messages = [
        {
            "role": "system",
            "content": "You are an expert at writing highly engaging titles."
        },
        {
            "role": "user",
            "content": f"Generate a high-engagement title for the following content:\n\n{example['content']}"
        },
        {
            "role": "assistant",
            "content": example["title"]
        }
    ]

    return {
        "text": tokenizer.apply_chat_template(
            messages,
            tokenize=False,
            add_generation_prompt=False
        ),
        "reward_score": example["reward_score"]
    }

train_ds = train_ds.map(
    format_example,
    remove_columns=[
        "content",
        "title",
        "metrics",
        "metadata",
    ],
    desc="Formatting train"
)

val_ds = val_ds.map(
    format_example,
    remove_columns=[
        "content",
        "title",
        "metrics",
        "metadata",
    ],
    desc="Formatting validation"
)

print(train_ds.features)
print(train_ds[0])

### Sequence Analysis
Analyzing the token length distribution of our training examples.

# Can we train the model to prefer high-engagement titles?

Standard training learns the *average* style of all titles. To optimize for engagement, we have three main choices:
1. **Filter:** Train only on the best titles (loses a lot of data).
2. **RLHF/DPO:** Advanced methods that compare good vs. bad titles (more complex).
3. **Weighted Training (Our Choice):** We keep all data but tell the model to pay more attention to titles with high `reward_score` values.

**Why Weighted SFT?**
It preserves all your data while ensuring a title with a 0.95 score influences the model much more than one with 0.10.

**The Implementation Plan:**
Since standard tools don't support "weighted loss" by default, we will:
1. **Keep** the `reward_score` in our dataset.
2. **Tokenize** the text for the model.
3. **Customize the Trainer:** Create a `WeightedTrainer` class that overrides the loss calculation.
4. **Apply Weights:** Multiply the error (loss) of each example by its `reward_score` so the model learns more from high-performing titles.

In [ ]:
lengths = []

for ex in train_ds.select(range(10000)):
    lengths.append(len(tokenizer(ex["text"]).input_ids))

import numpy as np

print("Mean:", np.mean(lengths))
print("Median:", np.median(lengths))
print("95%:", np.percentile(lengths, 95))
print("99%:", np.percentile(lengths, 99))
print("Max:", np.max(lengths))

### Tokenization
Converting the formatted text into input IDs and preparing the labels and rewards for the model.

In [ ]:
MAX_LENGTH = 512

def tokenize(example):
    enc = tokenizer(
        example["text"],
        truncation=True,
        max_length=MAX_LENGTH,
        padding=False,
    )

    enc["labels"] = enc["input_ids"].copy()
    enc["reward_score"] = example["reward_score"]

    return enc

train_tok = train_ds.map(
    tokenize,
    remove_columns=train_ds.column_names,
    desc="Tokenizing train"
)

val_tok = val_ds.map(
    tokenize,
    remove_columns=val_ds.column_names,
    desc="Tokenizing validation"
)

print(train_tok.features)
print(train_tok[0].keys())

### Data Collation
Setting up the data collator to handle batching and padding during training.

In [ ]:
from transformers import DataCollatorForLanguageModeling

data_collator = DataCollatorForLanguageModeling(
    tokenizer=tokenizer,
    mlm=False,
)

### Custom Loss Implementation
Defining a `WeightedTrainer` that scales loss based on the `reward_score` of each example.

In [ ]:
from transformers import Trainer
import torch
import torch.nn as nn

class WeightedTrainer(Trainer):
    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        reward = inputs.pop("reward_score").float()
        weights = 1.0 + reward

        labels = inputs["labels"]

        outputs = model(
            input_ids=inputs["input_ids"],
            attention_mask=inputs["attention_mask"],
            labels=labels,
        )

        logits = outputs.logits

        shift_logits = logits[..., :-1, :].contiguous()
        shift_labels = labels[..., 1:].contiguous()

        loss_fct = nn.CrossEntropyLoss(
            reduction="none",
            ignore_index=-100,
        )

        token_loss = loss_fct(
            shift_logits.view(-1, shift_logits.size(-1)),
            shift_labels.view(-1),
        )

        token_loss = token_loss.view(
            shift_labels.size(0),
            shift_labels.size(1),
        )

        valid_tokens = (shift_labels != -100).float()

        seq_loss = (
            (token_loss * valid_tokens).sum(dim=1)
            / valid_tokens.sum(dim=1).clamp(min=1)
        )

        loss = (seq_loss * weights).sum() / weights.sum()

        return (loss, outputs) if return_outputs else loss

### PEFT Configuration
Applying LoRA (Low-Rank Adaptation) to the model to allow efficient fine-tuning.

# **TRAINING ARGUMENTS**  ⚖

In [ ]:
from transformers import TrainingArguments

In [ ]:
training_args = TrainingArguments(
    output_dir="/content/drive/MyDrive/title_model",

    num_train_epochs=2,

    per_device_train_batch_size=1,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=16,

    learning_rate=2e-4,
    lr_scheduler_type="cosine",
    warmup_steps=50,

    logging_steps=20,

    eval_strategy="steps",
    eval_steps=1000,

    save_strategy="steps",
    save_steps=1000,
    save_total_limit=3,

    fp16=True,
    bf16=False,

    optim="paged_adamw_8bit",

    report_to="none",
    remove_unused_columns=False,
)

In [ ]:
from peft import LoraConfig, get_peft_model

lora_config = LoraConfig(
    r=16,
    lora_alpha=32,
    lora_dropout=0.05,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=[
        "q_proj",
        "k_proj",
        "v_proj",
        "o_proj",
        "gate_proj",
        "up_proj",
        "down_proj",
    ],
)

model = get_peft_model(model, lora_config)
model.print_trainable_parameters()

### Model Type Verification
Confirming that the model has been correctly wrapped as a `PeftModel`.

In [ ]:
from peft import PeftModel

print(isinstance(model, PeftModel))

In [ ]:
from peft import PeftModel

print(type(model))
print(isinstance(model, PeftModel))

### Execution
Launching the training process on a subset of the data.

# **TRAINING** ✅

In [ ]:
# Same 25k training examples
train_25k = train_tok.shuffle(seed=42).select(range(25000))

# Random 5k validation examples
val_5k = val_tok.shuffle(seed=42).select(range(5000))

trainer = WeightedTrainer(
    model=model,
    args=training_args,
    train_dataset=train_25k,
    eval_dataset=val_5k,
    data_collator=data_collator,
)

trainer.train(
    resume_from_checkpoint="/content/drive/MyDrive/colab_data/title_model/checkpoint-3000"
)

Step,Training Loss,Validation Loss


Step,Training Loss,Validation Loss
3126,23.593652,1.583161


TrainOutput(global_step=3126, training_loss=0.9091142885058031, metrics={'train_runtime': 1764.3409, 'train_samples_per_second': 28.339, 'train_steps_per_second': 1.772, 'total_flos': 4.868177889618125e+16, 'train_loss': 0.9091142885058031, 'epoch': 2.0})

### Persistence
Saving the fine-tuned LoRA weights and tokenizer locally.

In [ ]:
model.save_pretrained("/content/drive/MyDrive/title_model/final_lora")
tokenizer.save_pretrained("/content/drive/MyDrive/title_model/final_lora")

('/content/drive/MyDrive/title_model/final_lora/tokenizer_config.json',
 '/content/drive/MyDrive/title_model/final_lora/chat_template.jinja',
 '/content/drive/MyDrive/title_model/final_lora/tokenizer.json')

### Inference
Generating a title using the fine-tuned model to test its performance.

In [ ]:
messages = [
    {
        "role": "system",
        "content": "You are an expert at writing highly engaging titles."
    },
    {
        "role": "user",
        "content": """LinkedIn re-architected its distributed linear programming solver, DuaLip, by building a GPU-accelerated PyTorch version called DuaLip-PyTorch. The original Scala/Spark CPU-bound implementation struggled with extreme-scale optimization problems involving hundreds of millions of users and trillions of decision variables. The new system uses sparse tensor operations, batched projection kernels, and collective communication patterns (all-reduce, broadcast) to distribute computation across multiple GPUs. The result is a 75x speedup per iteration (8 GPUs vs. CPU baseline), near-linear multi-GPU scaling, and a more flexible, extensible solver architecture that bridges ML and optimization into a unified stack. The open-source implementation is available on GitHub.

"""}
]

text = tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = tokenizer(text, return_tensors="pt").to(model.device)

output = model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.1,
)

print(tokenizer.decode(output[0], skip_special_tokens=True))

system
You are an expert at writing highly engaging titles.
user
LinkedIn re-architected its distributed linear programming solver, DuaLip, by building a GPU-accelerated PyTorch version called DuaLip-PyTorch. The original Scala/Spark CPU-bound implementation struggled with extreme-scale optimization problems involving hundreds of millions of users and trillions of decision variables. The new system uses sparse tensor operations, batched projection kernels, and collective communication patterns (all-reduce, broadcast) to distribute computation across multiple GPUs. The result is a 75x speedup per iteration (8 GPUs vs. CPU baseline), near-linear multi-GPU scaling, and a more flexible, extensible solver architecture that bridges ML and optimization into a unified stack. The open-source implementation is available on GitHub.


assistant
DuaLip: A Scalable Linear Programming Solver for Extreme-Scale Optimization Problems


**We can see our model/LoRA adapter gives much better engaging titles compared to the basee model**

# **TESTING☕**

In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig
import torch

bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.float16,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_use_double_quant=True,
)

base_model = AutoModelForCausalLM.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct",
    quantization_config=bnb_config,
    device_map="auto",
)

base_tokenizer = AutoTokenizer.from_pretrained(
    "Qwen/Qwen2.5-1.5B-Instruct"
)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

In [ ]:
text = base_tokenizer.apply_chat_template(
    messages,
    tokenize=False,
    add_generation_prompt=True
)

inputs = base_tokenizer(text, return_tensors="pt").to(base_model.device)

output = base_model.generate(
    **inputs,
    max_new_tokens=40,
    temperature=0.7,
    do_sample=True,
    top_p=0.9,
    repetition_penalty=1.1,
)

print(base_tokenizer.decode(output[0], skip_special_tokens=True))

system
You are an expert at writing highly engaging titles.
user
LinkedIn re-architected its distributed linear programming solver, DuaLip, by building a GPU-accelerated PyTorch version called DuaLip-PyTorch. The original Scala/Spark CPU-bound implementation struggled with extreme-scale optimization problems involving hundreds of millions of users and trillions of decision variables. The new system uses sparse tensor operations, batched projection kernels, and collective communication patterns (all-reduce, broadcast) to distribute computation across multiple GPUs. The result is a 75x speedup per iteration (8 GPUs vs. CPU baseline), near-linear multi-GPU scaling, and a more flexible, extensible solver architecture that bridges ML and optimization into a unified stack. The open-source implementation is available on GitHub.


assistant
**"Revolutionizing Optimization: The NVIDIA GPU Accelerated DuaLip-PyTorch for Extreme-Scale Problems"**


In [ ]:
import torch
import pandas as pd

model.eval()

test_articles = [
    """
    Apple is developing a new generation of artificial intelligence tools
    designed to make its devices more useful and personalized. The company
    is expected to introduce new AI features across its iPhone, iPad and Mac
    products as competition in the technology industry intensifies.
    """,

    """
    A new study found that regular walking was associated with lower rates
    of cardiovascular disease among older adults. Researchers followed
    thousands of participants for several years and found that even moderate
    increases in daily activity were linked to better heart health.
    """,

    """
    Global oil prices rose on Monday after major producers signaled that
    they would maintain restrictions on output. Investors are watching
    demand forecasts closely as concerns about the global economy continue.
    """,

    """
    A major city has approved plans to expand its public transportation
    network, including several new rapid-transit lines. Officials said the
    project is intended to reduce congestion and provide commuters with
    faster alternatives to driving.
    """,

    """
    Scientists have developed a new battery design that could significantly
    increase energy storage capacity while reducing charging times. The
    researchers said the technology is still in the experimental stage but
    could eventually be used in electric vehicles.
    """
]


def generate_title(article):
    messages = [
        {
            "role": "system",
            "content": "You are an expert at writing highly engaging titles."
        },
        {
            "role": "user",
            "content": (
                "Generate a high-engagement title for the following content:\n\n"
                + article.strip()
            )
        }
    ]

    text = tokenizer.apply_chat_template(
        messages,
        tokenize=False,
        add_generation_prompt=True
    )

    inputs = tokenizer(
        text,
        return_tensors="pt"
    ).to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=40,
            do_sample=False
        )

    generated = output[0][inputs["input_ids"].shape[1]:]

    return tokenizer.decode(
        generated,
        skip_special_tokens=True
    ).strip()


results = []

for i, article in enumerate(test_articles, 1):

    # Base Qwen: temporarily disable LoRA
    with model.disable_adapter():
        base_title = generate_title(article)

    # Fine-tuned Qwen
    tuned_title = generate_title(article)

    results.append({
        "Article": i,
        "Base Qwen": base_title,
        "Fine-tuned Qwen": tuned_title
    })


df = pd.DataFrame(results)

display(df)

,Article,Base Qwen,Fine-tuned Qwen
0,1,"""Apple's AI Revolution: How the Tech Giant Pla...",Apple's New AI Tools Will Make Your Devices Mo...
1,2,"""Boost Your Heart Health: How Regular Walking ...",Walking Linked To Lower Risk Of Heart Disease ...
2,3,"""Oil Prices Soar: Producers Maintain Output Cu...",Oil Prices Rise on Expectations of OPEC Output...
3,4,"""City's Big Plan: New Rapid Transit Lines Set ...",City approves $1B plan to expand transit system
4,5,"""Revolutionizing Energy Storage: Scientists Un...",New battery design could double charging speed...


In [ ]:
for _, row in df.iterrows():
    print(f"\n{'='*80}")
    print(f"ARTICLE {row['Article']}")
    print(f"\nBASE QWEN:")
    print(row["Base Qwen"])
    print(f"\nFINE-TUNED:")
    print(row["Fine-tuned Qwen"])


ARTICLE 1

BASE QWEN:
"Apple's AI Revolution: How the Tech Giant Plans to Make Devices Smarter Than Ever Before"

FINE-TUNED:
Apple's New AI Tools Will Make Your Devices More Useful

ARTICLE 2

BASE QWEN:
"Boost Your Heart Health: How Regular Walking Can Lower Your Risk of Cardiovascular Disease Among Older Adults"

FINE-TUNED:
Walking Linked To Lower Risk Of Heart Disease In Older Adults

ARTICLE 3

BASE QWEN:
"Oil Prices Soar: Producers Maintain Output Cuts Amid Economic Worries"

FINE-TUNED:
Oil Prices Rise on Expectations of OPEC Output Cuts

ARTICLE 4

BASE QWEN:
"City's Big Plan: New Rapid Transit Lines Set to Revolutionize Commuting!"

FINE-TUNED:
City approves $1B plan to expand transit system

ARTICLE 5

BASE QWEN:
"Revolutionizing Energy Storage: Scientists Unveil Breakthrough Battery Design That Could Transform Electric Vehicles"

FINE-TUNED:
New battery design could double charging speed and boost energy storage


## **Some Social Media Posts🔔**

In [ ]:
test_social_posts = [
    """
    Just shipped the biggest update we've ever made to the app. Everything
    loads faster, search is finally instant, and we completely redesigned
    the dashboard. Been working on this for 8 months. It's live now.
    """,

    """
    I spent 6 months learning Python after work and today I finally got my
    first software engineering job. No CS degree, no bootcamp. Just a lot
    of late nights and building projects. Still can't believe it.
    """,

    """
    We just crossed 1 million users. When we launched three years ago,
    everyone told us the market was already too crowded. Huge thanks to
    everyone who gave us a chance.
    """,

    """
    Hot take: most people don't actually need a new productivity app.
    They need to stop checking Slack every five minutes and spend two hours
    doing uninterrupted work. The tools aren't the problem.
    """,

    """
    Our restaurant has been open for 27 years, and today my parents handed
    the keys to me. I'm the third generation to run the business. Same
    recipes, same building, completely different world.
    """,

    """
    This photo was taken exactly one year apart. On the left I had just
    started running. On the right I finished my first marathon. Never
    thought I'd make it this far.
    """,

    """
    We're cancelling our four-day workweek experiment after 18 months.
    Productivity went up, but customer response times got worse and the
    team ended up working longer hours on the remaining days.
    """,

    """
    AI just wrote a working version of a tool I spent two weeks building.
    I'm not sure whether to be impressed or terrified. The code wasn't
    perfect, but it was good enough to make me rethink how I work.
    """
]

# Generate base vs fine-tuned titles
results = []

for i, post in enumerate(test_social_posts, 1):

    with model.disable_adapter():
        base_title = generate_title(post)

    tuned_title = generate_title(post)

    results.append({
        "Post": i,
        "Base Qwen": base_title,
        "Fine-tuned Qwen": tuned_title
    })

social_df = pd.DataFrame(results)

for _, row in social_df.iterrows():
    print(f"\n{'='*80}")
    print(f"POST {row['Post']}")
    print(f"BASE:       {row['Base Qwen']}")
    print(f"FINE-TUNED: {row['Fine-tuned Qwen']}")


POST 1
BASE:       "Unleashed: The Epic Update That Speeds Up Your App & Redesigns How You See It!"
FINE-TUNED: We just updated the App Store with our best feature yet

POST 2
BASE:       "From Late Nights to Landing: My Journey from Learning Python to Software Engineering Success"
FINE-TUNED: My First Software Engineering Job

POST 3
BASE:       "Crossing the Million Mark: How Our Bold Launch Shaped the Future of [Your Platform/Service]"
FINE-TUNED: Our first year in business

POST 4
BASE:       "Stop Raving About Productivity Apps; It's Time to Focus on Real Life Habits"
FINE-TUNED: The Problem With Productivity Apps Is You, Not Them

POST 5
BASE:       "From Keys to Chef: My Journey in Running an Unforgettable Family Restaurant"
FINE-TUNED: The Restaurant My Parents Built

POST 6
BASE:       "From Running's Start to Finish: A Year of Determination and Triumph"
FINE-TUNED: One Year Ago, I Started Running. One Year Later, I Finished My First Marathon.

POST 7
BASE:       "From Four D

# **Deployment✌**

After fine-tuning the model with LoRA, the next step in deployment is to merge the LoRA adapter weights with the base model weights. This creates a single, self-contained model that can be easily deployed without needing the original base model and the adapter separately. This merged model will then be used for further conversion to GGUF format.

In [ ]:
from peft import PeftConfig
import os

adapter_path = "/content/drive/MyDrive/title_model/final_lora"

config = PeftConfig.from_pretrained(adapter_path)

print("Base model:", config.base_model_name_or_path)
print("PEFT type:", config.peft_type)
print("Task type:", config.task_type)
print("LoRA r:", config.r)
print("LoRA alpha:", config.lora_alpha)
print("LoRA dropout:", config.lora_dropout)
print("Target modules:", config.target_modules)

print("\nAdapter files:")
for f in sorted(os.listdir(adapter_path)):
    print(" ", f)

Base model: Qwen/Qwen2.5-1.5B-Instruct
PEFT type: PeftType.LORA
Task type: CAUSAL_LM
LoRA r: 16
LoRA alpha: 32
LoRA dropout: 0.05
Target modules: {'up_proj', 'k_proj', 'down_proj', 'gate_proj', 'o_proj', 'q_proj', 'v_proj'}

Adapter files:
  README.md
  adapter_config.json
  adapter_model.safetensors
  chat_template.jinja
  tokenizer.json
  tokenizer_config.json


In [ ]:
from transformers import AutoModelForCausalLM, AutoTokenizer
from peft import PeftModel
import torch

base_model = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/title_model/final_lora"
merged_path = "/content/drive/MyDrive/title_model/merged_model"

model = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype=torch.float16,
    device_map="auto",
)

model = PeftModel.from_pretrained(
    model,
    adapter_path,
)

model = model.merge_and_unload()

model.save_pretrained(
    merged_path,
    safe_serialization=True,
)

tokenizer = AutoTokenizer.from_pretrained(adapter_path)
tokenizer.save_pretrained(merged_path)

print("Merged model saved:", merged_path)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Merged model saved: /content/drive/MyDrive/title_model/merged_model


In [ ]:
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import PeftModel
import torch

base_model = "Qwen/Qwen2.5-1.5B-Instruct"
adapter_path = "/content/drive/MyDrive/title_model/final_lora"
merged_path = "/content/drive/MyDrive/title_model/merged_model"

prompt = """Generate a Highly Engaging Title for the Following Content:

Scientists have developed a new battery design that could significantly increase energy storage capacity while reducing charging time. The researchers say the design may eventually be useful for electric vehicles.

Headline:"""

def generate(model, tokenizer):
    inputs = tokenizer(prompt, return_tensors="pt").to(model.device)

    with torch.no_grad():
        output = model.generate(
            **inputs,
            max_new_tokens=30,
            do_sample=False,
        )

    return tokenizer.decode(
        output[0][inputs["input_ids"].shape[1]:],
        skip_special_tokens=True
    ).strip()


# ---- LoRA model ----
tokenizer_lora = AutoTokenizer.from_pretrained(adapter_path)

base = AutoModelForCausalLM.from_pretrained(
    base_model,
    dtype=torch.float16,
    device_map="auto",
)

lora_model = PeftModel.from_pretrained(
    base,
    adapter_path,
)

lora_model.eval()

lora_output = generate(lora_model, tokenizer_lora)


# ---- Merged model ----
tokenizer_merged = AutoTokenizer.from_pretrained(merged_path)

merged_model = AutoModelForCausalLM.from_pretrained(
    merged_path,
    dtype=torch.float16,
    device_map="auto",
)

merged_model.eval()

merged_output = generate(merged_model, tokenizer_merged)


print("LoRA model:")
print(lora_output)

print("\nMerged model:")
print(merged_output)

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/338 [00:00<?, ?it/s]

LoRA model:
Scientists develop new lithium-ion battery with 10 times higher energy density

The research team, led by Professor John Goodenough of the University of

Merged model:
Scientists develop new lithium-ion battery with 10 times higher energy density

The research team, led by Professor John Goodenough of the University of


### GGUF Conversion and Quantization

To make the model more efficient for deployment on various hardware, especially for local inference on CPUs, we convert the merged model into the GGUF (GGML Unified Format) format. This format is designed for fast and memory-efficient execution of large language models. Additionally, we quantize the model (e.g., to Q4_K_M) which reduces its size and computational requirements by storing weights with lower precision (e.g., 4-bit integers instead of 16-bit floats) while maintaining reasonable accuracy. This allows the model to run on devices with limited memory or computational power.

In [ ]:
%cd /content/llama.cpp

!rm -f "/content/merged_qwen_test.gguf"

!python convert_hf_to_gguf.py \
    "/content/drive/MyDrive/title_model/merged_model" \
    --outfile "/content/merged_qwen_test.gguf" \
    --outtype f16

/content/llama.cpp
INFO:hf-to-gguf:Loading model: merged_model
INFO:hf-to-gguf:Model architecture: Qwen2ForCausalLM
INFO:hf-to-gguf:gguf: indexing model part 'model.safetensors'
INFO:gguf.gguf_writer:gguf: This GGUF file is for Little Endian only
INFO:hf-to-gguf:Exporting model...
INFO:hf-to-gguf:token_embd.weight,         torch.float16 --> F16, shape = {1536, 151936}
INFO:hf-to-gguf:blk.0.attn_norm.weight,    torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.ffn_down.weight,     torch.float16 --> F16, shape = {8960, 1536}
INFO:hf-to-gguf:blk.0.ffn_gate.weight,     torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_up.weight,       torch.float16 --> F16, shape = {1536, 8960}
INFO:hf-to-gguf:blk.0.ffn_norm.weight,     torch.float16 --> F32, shape = {1536}
INFO:hf-to-gguf:blk.0.attn_k.bias,         torch.float16 --> F32, shape = {256}
INFO:hf-to-gguf:blk.0.attn_k.weight,       torch.float16 --> F16, shape = {1536, 256}
INFO:hf-to-gguf:blk.0.attn_output.weight

In [ ]:
%cd /content/llama.cpp

!./build/bin/llama-quantize \
    /content/merged_qwen_test.gguf \
    /content/NameGPT-Q4_K_M.gguf \
    Q4_K_M

/content/llama.cpp
llama_print_build_info: build = 1 (7ba604f)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/merged_qwen_test.gguf' to '/content/NameGPT-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from /content/merged_qwen_test.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              = 0.700000
llama_model_loader: - kv   5:          

In [ ]:
%cd /content/llama.cpp

!cmake -B build
!cmake --build build --config Release -j2

/content/llama.cpp
CMAKE_BUILD_TYPE=Release
-- Warning: ccache not found - consider installing it for faster compilation or disable this warning with GGML_CCACHE=OFF
-- CMAKE_SYSTEM_PROCESSOR: x86_64
-- GGML_SYSTEM_ARCH: x86
-- Including CPU backend
-- x86 detected
-- Adding CPU backend variant ggml-cpu: -march=native 
-- ggml version: 0.19.0
-- ggml commit:  7ba604f
-- OpenSSL found: 3.0.2
-- Generating embedded license file for target: llama-app
-- Configuring done (0.3s)
-- Generating done (0.7s)
-- Build files have been written to: /content/llama.cpp/build
[  1%] Built target llama-common-base
[  3%] Built target ggml-base
[  3%] Built target sha256
[  3%] Built target cpp-httplib
[  3%] Built target sha1
[  4%] Built target xxhash
[  4%] Built target llama-llava-cli
[  4%] Built target llama-ui-embed
[  4%] Built target llama-gemma3-cli
[  4%] Built target llama-minicpmv-cli
[  4%] Built target llama-qwen2vl-cli
[  5%] Provisioning UI assets
[  8%] Built target ggml-cpu
[  8%] Bui

In [ ]:
!rm -rf "/content/drive/MyDrive/title_model/llama.cpp"
!cp -r "/content/llama.cpp" "/content/drive/MyDrive/title_model/llama.cpp"

print("llama.cpp saved to Google Drive")

llama.cpp saved to Google Drive


In [ ]:
import os

base = "/content/drive/MyDrive/title_model/llama.cpp/build/bin"

for name in ["llama-cli", "llama-quantize"]:
    path = os.path.join(base, name)
    print(name, ":", "OK" if os.path.exists(path) else "MISSING")

llama-cli : OK
llama-quantize : OK


In [ ]:
# Copy the saved executable from Drive to local Colab storage
!cp "/content/drive/MyDrive/title_model/llama.cpp/build/bin/llama-quantize" \
    "/content/llama-quantize"

!chmod +x "/content/llama-quantize"

# Quantize directly from Drive → Drive
!"/content/llama-quantize" \
    "/content/drive/MyDrive/title_model/NameGPT-F16.gguf" \
    "/content/drive/MyDrive/title_model/NameGPT-Q4_K_M.gguf" \
    Q4_K_M

llama_print_build_info: build = 1 (7ba604f)
llama_print_build_info: built with GNU 11.4.0 for Linux x86_64
llama_quantize: quantizing '/content/drive/MyDrive/title_model/NameGPT-F16.gguf' to '/content/drive/MyDrive/title_model/NameGPT-Q4_K_M.gguf' as Q4_K_M
llama_model_loader: loaded meta data with 26 key-value pairs and 338 tensors from /content/drive/MyDrive/title_model/NameGPT-F16.gguf (version GGUF V3 (latest))
llama_model_loader: Dumping metadata keys/values. Note: KV overrides do not apply in this output.
llama_model_loader: - kv   0:                       general.architecture str              = qwen2
llama_model_loader: - kv   1:                               general.type str              = model
llama_model_loader: - kv   2:                     general.sampling.top_k i32              = 20
llama_model_loader: - kv   3:                     general.sampling.top_p f32              = 0.800000
llama_model_loader: - kv   4:                      general.sampling.temp f32              =

In [ ]:
import os

path = "/content/drive/MyDrive/title_model/NameGPT-Q4_K_M.gguf"

print("Exists:", os.path.exists(path))

if os.path.exists(path):
    print("Size:", round(os.path.getsize(path) / (1024**2), 2), "MiB")

Exists: True
Size: 940.37 MiB


In [ ]:
!cp "/content/drive/MyDrive/title_model/llama.cpp/build/bin/llama-cli" \
    "/content/llama-cli"

!chmod +x "/content/llama-cli"

!"/content/llama-cli" \
    -m "/content/drive/MyDrive/title_model/NameGPT-Q4_K_M.gguf" \
    -f "/content/prompt.txt" \
    -n 30 \
    --temp 0

cp: cannot create regular file '/content/llama-cli': Text file busy


Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/- 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b1-7ba604f
model      : /content/drive/MyDrive/title_model/NameGPT-Q4_K_M.gguf
ftype      : Q4_K - Medium
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat history
 

In [ ]:
!"/content/llama-cli" \
    -m "/content/drive/MyDrive/title_model/NameGPT-F16.gguf" \
    -f "/content/prompt.txt" \
    -n 30 \
    --temp 0



Loading model... |-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/-\|/ 

▄▄ ▄▄
██ ██
██ ██  ▀▀█▄ ███▄███▄  ▀▀█▄    ▄████ ████▄ ████▄
██ ██ ▄█▀██ ██ ██ ██ ▄█▀██    ██    ██ ██ ██ ██
██ ██ ▀█▄██ ██ ██ ██ ▀█▄██ ██ ▀████ ████▀ ████▀
                                    ██    ██
                                    ▀▀    ▀▀

build      : b1-7ba604f
model      : /content/drive/MyDrive/title_model/NameGPT-F16.gguf
ftype      : F16
modalities : text

available commands:
  /exit or Ctrl+C     stop or exit
  /regen              regenerate the last response
  /clear              clear the chat hist

In [ ]:
import os

model_path = "/content/drive/MyDrive/title_model/merged_model"

print("Merged model exists:", os.path.isdir(model_path))
print("Safetensors:", os.path.exists(os.path.join(model_path, "model.safetensors")))
print("Config:", os.path.exists(os.path.join(model_path, "config.json")))

Merged model exists: True
Safetensors: True
Config: True


In [ ]:
import json

config_path = "/content/drive/MyDrive/title_model/merged_model/config.json"

with open(config_path) as f:
    config = json.load(f)

print("model_type:", config.get("model_type"))
print("architectures:", config.get("architectures"))
print("hidden_size:", config.get("hidden_size"))
print("num_hidden_layers:", config.get("num_hidden_layers"))
print("num_attention_heads:", config.get("num_attention_heads"))

model_type: qwen2
architectures: ['Qwen2ForCausalLM']
hidden_size: 1536
num_hidden_layers: 28
num_attention_heads: 12


In [ ]:
import json

path = "/content/drive/MyDrive/title_model/merged_model/config.json"

with open(path) as f:
    config = json.load(f)

for key, value in config.items():
    if "rope" in key.lower():
        print(f"{key}: {value}")

rope_parameters: {'rope_theta': 1000000.0, 'rope_type': 'default'}


In [ ]:
!rm -f /content/namegpt_f16_webgpu.wasm

!python -m mlc_llm compile \
  /content/namegpt_mlc_f16_config/mlc-chat-config.json \
  --device webgpu \
  --host wasm32-unknown-unknown-wasm \
  --quantization q0f16 \
  --overrides "context_window_size=12288;prefill_chunk_size=2048;max_batch_size=1" \
  --output /content/namegpt_f16_webgpu.wasm